# E-Commerce Customer Intelligence Analysis

**Objective:** Analyze customer purchasing behavior to identify revenue drivers, high-value segments, product opportunities, and retention signals.

**Workflow:** CSV → Data Quality → Feature Engineering → Exploratory Analysis → SQL Business Analysis → Power BI

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_shopping_behavior.csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Rows: 3,900 | Columns: 18


In [2]:
# Quick data-quality review
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

   Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse  Clothing                     53   
1            2   19   Male        Sweater  Clothing                     64   
2            3   50   Male          Jeans  Clothing                     73   
3            4   21   Male        Sandals  Footwear                     90   
4            5   45   Male         Blouse  Clothing                     49   

        Location Size      Color  Season  Review Rating Subscription Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   Shipping Type Discount Applied Promo Code Used  Previ

In [3]:
# Fill missing review ratings using the median rating within each product category.
# Category-level imputation is preferred to a single global median because ratings can differ by category.
df["Review Rating"] = (
    df.groupby("Category")["Review Rating"]
      .transform(lambda s: s.fillna(s.median()))
)

# Standardize column names for SQL/Python readability.
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)
df = df.rename(columns={"purchase_amount_usd": "purchase_amount"})

print("Remaining missing values:", int(df.isna().sum().sum()))

Remaining missing values: 0


In [4]:
# Feature engineering
age_bins = [-1, 24, 34, 44, np.inf]
age_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

frequency_days = {
    "Weekly": 7,
    "Fortnightly": 14,
    "Bi-Weekly": 14,
    "Monthly": 30,
    "Quarterly": 90,
    "Every 3 Months": 90,
    "Annually": 365
}
df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_days)

df["customer_segment"] = pd.cut(
    df["previous_purchases"],
    bins=[0, 5, 20, np.inf],
    labels=["New / Low-Repeat", "Returning", "Loyal"]
)

# Promo-code usage duplicates discount status in this dataset, so keep one business field.
if "promo_code_used" in df.columns:
    df = df.drop(columns=["promo_code_used"])

df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days,customer_segment
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Senior,14,Returning
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14,New / Low-Repeat
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Senior,7,Loyal
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7,Loyal
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Senior,365,Loyal


## Executive KPIs

In [ ]:
kpis = {
    "Total Revenue": df["purchase_amount"].sum(),
    "Transactions": len(df),
    "Average Order Value": df["purchase_amount"].mean(),
    "Unique Customers": df["customer_id"].nunique(),
    "Subscribed Customers": (df["subscription_status"] == "Yes").sum(),
    "Discounted Orders": (df["discount_applied"] == "Yes").sum()
}
pd.Series(kpis).round(2)

## Revenue and Product Analysis

In [ ]:
category_summary = (
    df.groupby("category")
      .agg(
          revenue=("purchase_amount", "sum"),
          avg_order_value=("purchase_amount", "mean"),
          transactions=("customer_id", "count")
      )
      .sort_values("revenue", ascending=False)
)
category_summary.round(2)

In [ ]:
item_summary = (
    df.groupby("item_purchased")
      .agg(
          revenue=("purchase_amount", "sum"),
          avg_order_value=("purchase_amount", "mean"),
          transactions=("customer_id", "count"),
          avg_rating=("review_rating", "mean")
      )
      .sort_values("revenue", ascending=False)
)
item_summary.head(10).round(2)

## Customer Segmentation

In [ ]:
segment_summary = (
    df.groupby("customer_segment", observed=False)
      .agg(
          customers=("customer_id", "count"),
          revenue=("purchase_amount", "sum"),
          avg_order_value=("purchase_amount", "mean"),
          avg_previous_purchases=("previous_purchases", "mean")
      )
)
segment_summary["revenue_share_pct"] = (
    segment_summary["revenue"] / df["purchase_amount"].sum() * 100
)
segment_summary.round(2)

In [ ]:
subscription_summary = (
    df.groupby("subscription_status")
      .agg(
          customers=("customer_id", "count"),
          revenue=("purchase_amount", "sum"),
          avg_order_value=("purchase_amount", "mean"),
          avg_previous_purchases=("previous_purchases", "mean")
      )
)
subscription_summary["revenue_share_pct"] = (
    subscription_summary["revenue"] / df["purchase_amount"].sum() * 100
)
subscription_summary.round(2)

## Discount Effectiveness

In [ ]:
discount_summary = (
    df.groupby("discount_applied")
      .agg(
          orders=("customer_id", "count"),
          revenue=("purchase_amount", "sum"),
          avg_order_value=("purchase_amount", "mean")
      )
)
discount_summary.round(2)

In [ ]:
# Products with the highest discount adoption
discount_by_item = (
    df.groupby("item_purchased")
      .agg(
          orders=("customer_id", "count"),
          discounted_orders=("discount_applied", lambda s: (s == "Yes").sum())
      )
)
discount_by_item["discount_rate_pct"] = (
    discount_by_item["discounted_orders"] / discount_by_item["orders"] * 100
)
discount_by_item.sort_values("discount_rate_pct", ascending=False).head(10).round(2)

## Customer Behavior by Season, Shipping and Age

In [ ]:
season_summary = (
    df.groupby("season")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
season_summary.round(2)

In [ ]:
shipping_summary = (
    df.groupby("shipping_type")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"))
      .sort_values("avg_order_value", ascending=False)
)
shipping_summary.round(2)

In [ ]:
age_summary = (
    df.groupby("age_group", observed=False)
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"))
)
age_summary.round(2)

## Analyst Takeaways

The analysis is intentionally focused on actionable business questions: where revenue comes from, which customer segments deserve retention investment, where discounts are concentrated, and which products/categories should receive attention. SQL contains the reproducible business queries and Power BI is used for interactive reporting.